# Analysis pipeline: one patient in, every result out (M3-T05)

One command, `scripts/analyze.py`, runs every Milestone 3 step on all scans of one patient:

| Output | Step (task) | File in `OUT` |
| --- | --- | --- |
| tumor mask | seg-model v1, nnU-Net (M2) | `<scan>/<scan>_seg.nii.gz` |
| confidence map | 8 mirror passes (M3-T04) | `<scan>/<scan>_uncertainty.nii.gz`, `_disagreement.nii.gz`, `uncertainty.csv` |
| volumes per region | M3-T01 | `volumes.csv` |
| change between scans | M3-T02 (patients with 2+ scans) | `changes.csv` |
| radiomics features | M3-T03 | `radiomics.csv` |
| check against the expert mask | Dice, only if the scan has one | `metrics.csv` |

Everything is also recorded in `run_summary.json`: the model, the settings, the repo commit and the time taken.
The code is in `src/bmts/pipeline/analyze.py` and its tests in `tests/pipeline/`.

**Safety:** the patient is checked against `qc/split_v1.csv` first. Locked test patients and the external cohorts
(BraTS-Africa, UTSW) are refused, because they are kept for the final evaluation (M5).

**How to run:**
1. **Runtime → Change runtime type → T4 GPU.**
2. Set `DATASET` and `SUBJECT` in cell 1.
3. **Runtime → Run all.**

It takes a few minutes per scan. Results are written to `BrainMRI_Data/runs/M3-T05_pipeline/<subject>/`.

> Research prototype for decision support. The numbers describe the model's masks; they are not a diagnosis.

*This replaces the first version of this notebook, `08_unified_brain_mri_pipeline.ipynb`, which is still in the git
history (commit 19c00a4). Its steps now call the team's tested code, and the confidence map is M3-T04's.*

In [ ]:
# 1. Settings and Drive
DATASET = "rhuh"            # brats2021, rhuh or mu
SUBJECT = "RHUH-0002"       # e.g. RHUH-0002 (3 scans: pre-op, early post-op, recurrence) or BraTS2021_00032
DEVICE = "cuda"             # "cpu" also works, but is slow
RADIOMICS = True
BRANCH = "main"

import os, sys, json, hashlib, subprocess
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
BASE = Path("/content/drive/MyDrive/BrainMRI_Data")
SPLIT_CSV = BASE / "qc/split_v1.csv"
MODEL_DIR = BASE / "runs/M2-T04_nnunet/nnUNet_results/Dataset501_BraTS2021/nnUNetTrainer_50epochs__nnUNetPlans__3d_fullres"
OUT = BASE / "runs/M3-T05_pipeline" / SUBJECT
!nvidia-smi --query-gpu=name --format=csv,noheader || echo "no GPU: set DEVICE = 'cpu' or change the runtime type"

In [ ]:
# 2. Code from the team repo and the pinned packages
if Path("/content/repo/.git").exists():
    !git -C /content/repo fetch -q origin && git -C /content/repo checkout -q {BRANCH} && git -C /content/repo pull -q origin {BRANCH}
else:
    !git clone -q -b {BRANCH} https://github.com/nhahub/NHA-5-187.git /content/repo
!git -C /content/repo log --oneline -1
!pip -q install "nnunetv2==2.8.1" "monai>=1.4"
if RADIOMICS:
    !pip -q install -r /content/repo/environment/requirements-radiomics.txt
for k in ("nnUNet_raw", "nnUNet_preprocessed", "nnUNet_results"):   # nnU-Net warns when these are unset
    os.environ.setdefault(k, "/content/nnunet_unused")
sys.path.insert(0, "/content/repo/src")

In [ ]:
# 3. Is this patient allowed? (locked test patients and external cohorts are refused) + is the model the registered one?
from bmts.data.splits import check_subject
print(DATASET, SUBJECT, "->", check_subject(SPLIT_CSV, DATASET, SUBJECT))

manifest = json.load(open("/content/repo/models/registry/seg-model/v1/manifest.json"))
ckpt = MODEL_DIR / "fold_0" / manifest["nnunet"]["checkpoint"]
h = hashlib.sha256()
with open(ckpt, "rb") as f:
    for block in iter(lambda: f.read(1 << 24), b""):
        h.update(block)
assert h.hexdigest() == manifest["checkpoint_file"]["sha256"], "checkpoint differs from the registered seg-model v1"
print("checkpoint OK:", ckpt.name)

In [ ]:
# 4. Cleaned scans of this patient only (M1-T04), in /content/processed/<dataset>/<subject>/<timepoint>/
processed = Path("/content/processed") / DATASET / SUBJECT
if not processed.exists():
    if DATASET == "brats2021":   # the cleaned train + validation archive (about 10 minutes)
        !mkdir -p /content/processed && tar -xf "{BASE}/processed/brats2021_trainval_v1.tar" -C /content/processed "brats2021/{SUBJECT}"
    else:                        # clean just this patient with the team's cleaning step
        from bmts.data.cleaning import standardize as S
        S.ensure_extracted(DATASET)
        jobs = S.run([DATASET], subjects={SUBJECT})
        assert (jobs.status != "failed").all(), jobs[jobs.status == "failed"]
print(sorted(p.name for p in processed.iterdir()))

In [ ]:
# 5. The whole analysis: one command
%cd /content/repo
radiomics_arg = "" if RADIOMICS else "--no-radiomics"
!python scripts/analyze.py --dataset {DATASET} --subject {SUBJECT} --out "{OUT}" --device {DEVICE} {radiomics_arg}

In [ ]:
# 6. Results
import pandas as pd
from IPython.display import display
for name in ("volumes", "uncertainty", "changes", "metrics"):
    path = OUT / f"{name}.csv"
    if path.exists():
        print(name); display(pd.read_csv(path).round(3))
if (OUT / "radiomics.csv").exists():
    rad = pd.read_csv(OUT / "radiomics.csv")
    print("radiomics:", rad.shape, "| quality notes:", rad.quality_note.dropna().unique().tolist())
print(json.dumps(json.load(open(OUT / "run_summary.json")), indent=1))

In [ ]:
# 7. A picture per scan: FLAIR, predicted mask and confidence map on the slice with the most tumor
import numpy as np, nibabel as nib, matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
summary = json.load(open(OUT / "run_summary.json"))
scans = summary["scans"]
fig, axes = plt.subplots(len(scans), 3, figsize=(10, 3.4 * len(scans)), squeeze=False)
for row, scan in zip(axes, scans):
    folder = next(p for p in sorted(processed.iterdir()) if scan in (SUBJECT, f"{SUBJECT}_tp{p.name}"))
    flair = nib.load(folder / "flair.nii.gz").get_fdata()
    seg = np.asarray(nib.load(OUT / scan / f"{scan}_seg.nii.gz").dataobj)
    unc = nib.load(OUT / scan / f"{scan}_uncertainty.nii.gz").get_fdata()
    z = int((seg > 0).sum((0, 1)).argmax())
    for ax, title in zip(row, ("FLAIR", "predicted mask", "confidence map (uncertainty)")):
        ax.imshow(np.rot90(flair[:, :, z]), cmap="gray"); ax.set_axis_off(); ax.set_title(f"{scan}: {title}", fontsize=9)
    row[1].imshow(np.rot90(np.ma.masked_equal(seg[:, :, z], 0)), cmap=ListedColormap(["#eb6834", "#1baf7a", "#e87ba4"]),
                  vmin=1, vmax=3, alpha=0.6, interpolation="nearest")
    im = row[2].imshow(np.rot90(np.ma.masked_less(unc[:, :, z], 0.02)), cmap="autumn_r", vmin=0, vmax=1)
fig.colorbar(im, ax=axes[:, 2].tolist(), shrink=0.6, label="uncertainty (0 = sure)")
plt.savefig(OUT / "overview.png", dpi=120, bbox_inches="tight"); plt.show()

In [ ]:
# 8. Finish writing to Drive
drive.flush_and_unmount()